# ASL Fingerspelling Recognition
CSS 324 Final Project - classifying ASL hand signs from images using MediaPipe hand landmarks + ML.

We're covering letters A-Y (skipping J and Z since those require tracking motion, not just a single frame).

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/asl_project/data', exist_ok=True)

Mounted at /content/drive


In [ ]:
from google.colab import files
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle-2.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

# make sure it works
!kaggle datasets list --search 'asl alphabet'

Saving kaggle-2.json to kaggle-2 (1).json
ref                                                          title                                                    size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-----------------------------------------------------------  ------------------------------------------------  -----------  --------------------------  -------------  ---------  ---------------  
grassknoted/asl-alphabet                                     ASL Alphabet                                       1100887034  2018-04-22 19:31:36.210000         116049       1244  0.875            
lexset/synthetic-asl-alphabet                                Synthetic ASL Alphabet                             7067002276  2022-06-17 20:36:37.220000           5407         57  0.9375           
danrasband/asl-alphabet-test                                 ASL Alphabet Test                                    25489344  2018-08-01 04:02:18.950000           5340         

In [ ]:
# main dataset
!kaggle datasets download -d grassknoted/asl-alphabet -p /content/drive/MyDrive/asl_project/data/ --unzip

# second dataset - different capture conditions, adds some variety
!kaggle datasets download -d mrgeislinger/asl-rgb-depth-fingerspelling-spelling-it-out -p /content/drive/MyDrive/asl_project/data/ --unzip

Dataset URL: https://www.kaggle.com/datasets/grassknoted/asl-alphabet
License(s): GPL-2.0
100% 1.03G/1.03G [00:36<00:00, 29.8MB/s]

Dataset URL: https://www.kaggle.com/datasets/mrgeislinger/asl-rgb-depth-fingerspelling-spelling-it-out
License(s): unknown
100% 2.11G/2.11G [01:43<00:00, 21.8MB/s]


In [ ]:
!kaggle datasets download -d mrgeislinger/asl-rgb-depth-fingerspelling-spelling-it-out \
    -p /content/drive/MyDrive/asl_project/data/ --unzip

Dataset URL: https://www.kaggle.com/datasets/mrgeislinger/asl-rgb-depth-fingerspelling-spelling-it-out
License(s): unknown
100% 2.11G/2.11G [02:17<00:00, 16.5MB/s]


In [ ]:
import os

base = '/content/drive/MyDrive/asl_project/data/'
for item in os.listdir(base):
    path = os.path.join(base, item)
    if os.path.isdir(path):
        count = sum(len(files) for _, _, files in os.walk(path))
        print(f'{item}/  →  {count} files')
    else:
        size_mb = os.path.getsize(path) / 1024 / 1024
        print(f'{item}  →  {size_mb:.1f} MB')

asl_alphabet_test/  →  28 files
asl_alphabet_train/  →  87000 files
dataset5/  →  131668 files


In [ ]:
from pathlib import Path

base = Path('/content/drive/MyDrive/asl_project/data/')

for ds in ['asl_alphabet_train', 'asl_alphabet_test', 'dataset5']:
    p = base / ds
    subdirs = sorted([f.name for f in p.iterdir() if f.is_dir()])
    direct_files = [f.name for f in p.iterdir() if f.is_file()]

    print(f'\n--- {ds} ---')
    print(f'Subfolders ({len(subdirs)}): {subdirs[:10]}')
    print(f'Direct files: {direct_files[:5]}')

    if subdirs:
        first = p / subdirs[0]
        samples = list(first.iterdir())[:3]
        print(f"Inside '{subdirs[0]}/': {[s.name for s in samples]}")


--- asl_alphabet_train ---
Subfolders (1): ['asl_alphabet_train']
Direct files: []
Inside 'asl_alphabet_train/': ['A', 'B', 'C']

--- asl_alphabet_test ---
Subfolders (1): ['asl_alphabet_test']
Direct files: []
Inside 'asl_alphabet_test/': ['A_test.jpg', 'B_test.jpg', 'C_test.jpg']

--- dataset5 ---
Subfolders (5): ['A', 'B', 'C', 'D', 'E']
Direct files: []
Inside 'A/': ['a', 'b', 'c']


In [ ]:
!pip install mediapipe

In [ ]:
import urllib.request
urllib.request.urlretrieve(
    'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task',
    '/content/hand_landmarker.task'
)
print('Model downloaded')

Model downloaded


In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import pandas as pd
import random
from pathlib import Path

train_root = Path('/content/drive/MyDrive/asl_project/data/asl_alphabet_train/asl_alphabet_train')
LETTERS = list('ABCDEFGHIKLMNOPQRSTUVWXY')
CAP = 3000

records = []
failed = 0
total = 0

options = vision.HandLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path='/content/hand_landmarker.task'),
    num_hands=1,
    min_hand_detection_confidence=0.5
)

def get_landmarks(img_path, detector):
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    result = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
    if not result.hand_landmarks:
        return None
    lm = result.hand_landmarks[0]
    wx, wy, wz = lm[0].x, lm[0].y, lm[0].z
    row = {}
    for i, pt in enumerate(lm):
        row[f'x{i}'] = pt.x - wx
        row[f'y{i}'] = pt.y - wy
        row[f'z{i}'] = pt.z - wz
    return row

print('Processing dataset1...')
with vision.HandLandmarker.create_from_options(options) as detector:
    for letter in LETTERS:
        folder = train_root / letter
        if not folder.exists():
            print(f'  {letter}: missing, skipping')
            continue
        imgs = list(folder.glob('*.jpg'))
        random.shuffle(imgs)
        before = len(records)
        for path in imgs[:CAP]:
            total += 1
            row = get_landmarks(path, detector)
            if row is None:
                failed += 1
                continue
            row['letter'] = letter
            row['source'] = 'kaggle_alphabet'
            records.append(row)
        print(f'  {letter}: {len(records) - before} extracted')

df = pd.DataFrame(records)
out = '/content/drive/MyDrive/asl_project/data/landmarks_dataset1.csv'
df.to_csv(out, index=False)

print(f'\nTotal: {len(df)} rows')
print(f'Dropout: {failed}/{total} ({failed/total*100:.1f}%)')
print(f'Saved to: {out}')
print(f'\nClass distribution:\n{df["letter"].value_counts().sort_index()}')

Processing dataset1...
  A: 363 extracted
  B: 363 extracted
  C: 312 extracted
  D: 416 extracted
  E: 379 extracted
  F: 476 extracted
  G: 408 extracted
  H: 395 extracted
  I: 393 extracted
  K: 448 extracted
  L: 416 extracted
  M: 272 extracted
  N: 223 extracted
  O: 376 extracted
  P: 352 extracted
  Q: 340 extracted
  R: 426 extracted
  S: 425 extracted
  T: 389 extracted
  U: 423 extracted
  V: 434 extracted
  W: 402 extracted
  X: 353 extracted
  Y: 441 extracted

Total: 9225 rows
Dropout: 2775/12000 (23.1%)
Saved to: /content/drive/MyDrive/asl_project/data/landmarks_dataset1.csv

Class distribution:
letter
A    363
B    363
C    312
D    416
E    379
F    476
G    408
H    395
I    393
K    448
L    416
M    272
N    223
O    376
P    352
Q    340
R    426
S    425
T    389
U    423
V    434
W    402
X    353
Y    441
Name: count, dtype: int64


In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import pandas as pd
import random
import shutil
from pathlib import Path
from tqdm import tqdm
import time

base = Path('/content/drive/MyDrive/asl_project/data/')
drive_train = base / 'asl_alphabet_train/asl_alphabet_train'
local_train = Path('/content/asl_local/asl_alphabet_train')  # drive reads are slow
model_path = '/content/hand_landmarker.task'

CLASSES = list('ABCDEFGHIKLMNOPQRSTUVWXY') + ['space', 'del', 'nothing']
label_map = {'space': 'SPACE', 'del': 'DELETE', 'nothing': 'NOTHING'}
CAP = 1000
random.seed(42)

# copy from drive to local first - about 5x faster for extraction
def copy_dataset():
    if local_train.exists():
        n = len([d for d in local_train.iterdir() if d.is_dir()])
        if n >= len(CLASSES):
            print(f'local copy already there ({n} folders), skipping')
            return
        print(f'incomplete ({n} folders), re-copying...')
        shutil.rmtree(local_train.parent, ignore_errors=True)
    print('copying dataset to local storage, this takes a while...')
    local_train.mkdir(parents=True, exist_ok=True)
    for cls in tqdm(CLASSES, desc='copying classes'):
        src = drive_train / cls
        dst = local_train / cls
        if src.exists() and not dst.exists():
            shutil.copytree(src, dst)
    print('done\n')

def get_landmarks(img_path, detector):
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    res = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
    if not res.hand_landmarks:
        return None
    lm = res.hand_landmarks[0]
    wx, wy, wz = lm[0].x, lm[0].y, lm[0].z
    row = {}
    for i, pt in enumerate(lm):
        row[f'x{i}'] = pt.x - wx
        row[f'y{i}'] = pt.y - wy
        row[f'z{i}'] = pt.z - wz
    return row

copy_dataset()

options = vision.HandLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path=model_path),
    num_hands=1,
    min_hand_detection_confidence=0.5
)

records = []
attempted = 0
t0 = time.time()

with vision.HandLandmarker.create_from_options(options) as detector:
    for cls in CLASSES:
        folder = local_train / cls
        if not folder.exists():
            print(f'missing: {cls}')
            continue

        label = label_map.get(cls, cls)
        imgs = list(folder.glob('*.jpg')) + list(folder.glob('*.png'))
        random.shuffle(imgs)
        imgs = imgs[:CAP]
        attempted += len(imgs)

        for path in tqdm(imgs, desc=f'{label:>8}'):
            row = get_landmarks(path, detector)
            if row is None:
                continue
            row['letter'] = label
            row['source'] = 'kaggle_alphabet'
            records.append(row)

out_csv = base / 'landmarks_dataset1.csv'
df = pd.DataFrame(records)
df.to_csv(out_csv, index=False)

elapsed = time.time() - t0
dropped = attempted - len(df)
print(f'Saving to {out_csv}...')
print()
print(f'Total Extracted:  {len(df)} rows')
print(f'Dropout Rate:     {dropped}/{attempted} ({dropped/attempted*100:.1f}%)')
print(f'Overall Runtime:  {elapsed:.2f} seconds')
print()
print(df['letter'].value_counts().sort_index())

Incomplete local copy (3 folders). Re-copying...
Copying dataset to local Colab storage (this drastically speeds up extraction)...


Copying classes: 100%|██████████| 27/27 [49:27<00:00, 109.90s/it]


Local copy ready in 2967.42 seconds.

Extracting landmarks | 27 classes | CAP=1000



   NOTHING: 100%|██████████| 1000/1000 [00:21<00:00, 47.52it/s]



Saving to /content/drive/MyDrive/asl_project/data/landmarks_dataset1.csv...

Total Extracted:  19592 rows
Dropout Rate:     7408/27000 (27.4%)
Overall Runtime:  3782.82 seconds

letter
A         721
B         754
C         662
D         811
DELETE    588
E         756
F         961
G         819
H         786
I         787
K         892
L         857
M         538
N         429
O         740
P         671
Q         713
R         855
S         855
SPACE     503
T         777
U         841
V         860
W         839
X         720
Y         857
Name: count, dtype: int64


## Self-Collected Data

To supplement the Kaggle dataset we each recorded our own samples using a custom collection script (`collect_asl_samples.py`). The script uses MediaPipe to confirm a hand is visible before saving each frame, so nearly all images extract successfully.

- 100 images × 27 classes per person
- Landmarks extracted with the same pipeline as the Kaggle data (wrist-normalized, 63 features)
- Source column set to `self_collected_[name]` to track origin

After merging, the combined dataset covers more hand shapes, skin tones, and lighting conditions than the Kaggle data alone.

## Merge All Landmark CSVs

Teammates uploaded their self-collected CSVs to the shared Drive folder. We load all of them together with the Kaggle-extracted data and save one merged file that the rest of the notebook works from.

In [2]:
import pandas as pd
from pathlib import Path

base = Path('/content/drive/MyDrive/asl_project/data/')

files = sorted(base.glob('landmarks_*.csv'))
print(f'found {len(files)} files:')
for f in files:
    print(f'  {f.name}')

dfs = [pd.read_csv(f) for f in files]
df = pd.concat(dfs, ignore_index=True)

df.to_csv(base / 'landmarks_merged.csv', index=False)

print(f'\nsource breakdown:')
print(df['source'].value_counts().to_string())
print(f'\ntotal rows: {len(df)}')
print(f'classes: {sorted(df["letter"].unique())}')

found 6 files:
  landmarks_akezhun.csv
  landmarks_akira.csv
  landmarks_dataset1.csv
  landmarks_i2.csv
  landmarks_pasha.csv
  landmarks_pavel.csv

source breakdown:
source
kaggle_alphabet           19592
self_collected_akezhun     5200
self_collected_akira       2600
self_collected_i2          2600
self_collected_pavel       2600
self_collected_pasha       1300

total rows: 33892
classes: ['A', 'B', 'C', 'D', 'DELETE', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'SPACE', 'T', 'U', 'V', 'W', 'X', 'Y']
